# QR101B vs GDM-8341

Reads the validation run from its suitcase file in `data/` and plots the GDM-8341 against the QR101B. `qr101b_gdm8341_validation.py` wrote the file. Each line of it is one event-model document.

The notebook writes no files. To plot another run, change `RUN_UID`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cryojane_bench.plotting import BLUE, INK, MUTED, ORANGE, apply_style
from cryojane_bench.runs import load_run
from qr101b_gdm8341_validation import summarize

apply_style()
RED = "#d03b3b"
RUN_UID = "d80a63b6"
MISSED_LIMIT_OHM = 0.1  # The QR10x manual allows under one nominal step between SP and PV.
GDM_SPEC_FRACTION = 0.001  # GDM-834X resistance accuracy: 0.1% of reading...
GDM_SPEC_COUNTS = 3  # ...plus 3 counts.
GDM_FULL_SCALE_COUNTS = 50_000  # A count on the 50 kohm range is 1 ohm.

path = next(Path("data").glob(f"*_{RUN_UID}.jsonl"))
run = load_run(path)
config = run.configuration()
box, meter = config["qr101b"], config["gdm"]
readings = run.table()

print(f"Run {run.start['uid']}, plan {run.start['plan_name']}, exit {run.stop['exit_status']}")
print(f"Wiring: {run.start['wiring']}")
print(f"{box['qr101b_device_type']} S/N {box['qr101b_serial_number']}, firmware {box['qr101b_firmware']}, limit {box['qr101b_limit']} ohm")
print(f"{meter['gdm_model']} S/N {meter['gdm_serial_number']}, firmware {meter['gdm_firmware']}, rate {meter['gdm_rate']}")
print(f"{readings.index[0]} to {readings.index[-1]}, {len(readings)} readings")
print(f"Box temperature {readings.qr101b_internal_temperature.min():.2f} to {readings.qr101b_internal_temperature.max():.2f} °C")


## One row per step

Each step is the median of the readings taken at that setpoint. `missed` marks steps where the box's calculated value (PV) is further from its setpoint than the manual allows. `gdm_limit_ppm` is the meter's accuracy limit at that reading.


In [ ]:
steps = summarize(path).reset_index()
steps["box_error_ohm"] = steps.calculated_ohm - steps.setpoint_ohm
steps["missed"] = steps.box_error_ohm.abs() > MISSED_LIMIT_OHM
count_ohm = steps.gdm_range_ohm / GDM_FULL_SCALE_COUNTS
steps["gdm_limit_ppm"] = 1e6 * (GDM_SPEC_FRACTION + GDM_SPEC_COUNTS * count_ohm / steps.calculated_ohm)
missed, good = steps[steps.missed], steps[~steps.missed]

worst = missed.loc[missed.box_error_ohm.abs().idxmax()]
below = good.setpoint_ohm[good.setpoint_ohm < missed.setpoint_ohm.min()].max()
above = good.setpoint_ohm[good.setpoint_ohm > missed.setpoint_ohm.max()].min()
print(f"Box missed its setpoint on {len(missed)} of {len(steps)} steps")
print(f"Good setpoints either side: {below:,.0f} and {above:,.0f} ohm")
print(f"Largest miss: {worst.box_error_ohm:+,.1f} ohm at {worst.setpoint_ohm:,.0f} ohm set ({100 * worst.box_error_ohm / worst.setpoint_ohm:+.1f}%)")
print(f"Largest PV - setpoint on the other steps: {good.box_error_ohm.abs().max():.3f} ohm")
print(f"GDM vs PV: {steps.gdm_vs_pv_ppm.min():+.0f} to {steps.gdm_vs_pv_ppm.max():+.0f} ppm")
print(f"GDM inside its accuracy limit on {(steps.gdm_vs_pv_ppm.abs() <= steps.gdm_limit_ppm).sum()} of {len(steps)} steps")


## GDM-8341 reading against QR101B setpoint

The meter leaves the setpoint line only where the box misses. Up and down sweeps overlap.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
top = 1.05 * max(steps.setpoint_ohm.max(), steps.gdm_median_ohm.max())
ax.plot([0, top], [0, top], color=MUTED, linestyle="--", linewidth=1.5, label="Reading equals setpoint", zorder=1)
ax.plot(good.setpoint_ohm, good.gdm_median_ohm, "o", color=BLUE, markersize=7, markeredgecolor="#fcfcfb", label=f"Box within {MISSED_LIMIT_OHM} Ω of setpoint", zorder=2)
ax.plot(missed.setpoint_ohm, missed.gdm_median_ohm, "o", color=RED, markersize=7, markeredgecolor="#fcfcfb", label="Box missed setpoint", zorder=3)

stuck = missed[missed.box_error_ohm > 0].sort_values("setpoint_ohm").iloc[0]
low = missed.loc[missed.box_error_ohm.idxmin()]
ax.annotate(f"Reads about {stuck.gdm_median_ohm:,.0f} Ω from {stuck.setpoint_ohm:,.0f} Ω up", (stuck.setpoint_ohm, stuck.gdm_median_ohm), xytext=(-10, 8), textcoords="offset points", ha="right", color=INK)
ax.annotate(f"{low.setpoint_ohm:,.0f} Ω set, {low.gdm_median_ohm:,.0f} Ω read", (low.setpoint_ohm, low.gdm_median_ohm), xytext=(10, -14), textcoords="offset points", color=INK)

ax.set(xlim=(0, top), ylim=(0, top), xlabel="QR101B setpoint (Ω)", ylabel="GDM-8341 reading (Ω)")
ax.xaxis.set_major_formatter("{x:,.0f}")
ax.yaxis.set_major_formatter("{x:,.0f}")
ax.set_title("The meter leaves the setpoint line only where the box misses")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()


## The box's error and the meter's error

Left: the box holds one value across the upper part of the band, so its error falls steadily as the setpoint rises towards that value. Both sweep directions gave the same PV, so only the up sweep is drawn.

Right: the meter reads a steady amount low. PV is calculated, so this compares the two instruments and does not show which is right.


In [ ]:
fig, (ax_box, ax_meter) = plt.subplots(1, 2, figsize=(12, 5))

up = steps[steps.direction == "up"].sort_values("setpoint_ohm")
ax_box.plot(up.setpoint_ohm, up.box_error_ohm, color=MUTED, zorder=1)
ax_box.plot(up.setpoint_ohm[~up.missed], up.box_error_ohm[~up.missed], "o", color=BLUE, markersize=7, markeredgecolor="#fcfcfb", zorder=2)
ax_box.plot(up.setpoint_ohm[up.missed], up.box_error_ohm[up.missed], "o", color=RED, markersize=7, markeredgecolor="#fcfcfb", zorder=3)
ax_box.set(xlabel="QR101B setpoint (Ω)", ylabel="PV − setpoint (Ω)")
ax_box.xaxis.set_major_formatter("{x:,.0f}")
ax_box.yaxis.set_major_formatter("{x:+,.0f}")
ax_box.set_title("The box holds one value across the upper band")

for direction, color in (("up", BLUE), ("down", ORANGE)):
    sweep = steps[steps.direction == direction].sort_values("calculated_ohm")
    ax_meter.plot(sweep.calculated_ohm, sweep.gdm_vs_pv_ppm, color=color, label=f"{direction.capitalize()} sweep")
limit = steps.sort_values("calculated_ohm")
ax_meter.plot(limit.calculated_ohm, -limit.gdm_limit_ppm, color=MUTED, linestyle="--", linewidth=1.5, label="GDM-8341 accuracy limit")
ax_meter.set(xlabel="QR101B PV (Ω)", ylabel="GDM vs PV (ppm)", ylim=(-1.05 * limit.gdm_limit_ppm.max(), 0))
ax_meter.xaxis.set_major_formatter("{x:,.0f}")
ax_meter.yaxis.set_major_formatter("{x:,.0f}")
ax_meter.set_title("The meter reads a steady amount low")
ax_meter.legend(loc="lower right")
fig.tight_layout()
plt.show()


## Every step, in the order it ran


In [ ]:
table = pd.DataFrame({
    "sweep": steps.direction,
    "setpoint (Ω)": steps.setpoint_ohm,
    "box PV (Ω)": steps.calculated_ohm,
    "GDM-8341 (Ω)": steps.gdm_median_ohm,
    "PV − setpoint (Ω)": steps.box_error_ohm,
    "GDM vs PV (ppm)": steps.gdm_vs_pv_ppm,
    "box": np.where(steps.missed, "Missed", "OK"),
})
table.index = pd.RangeIndex(1, len(table) + 1, name="step")
print(table.round({"box PV (Ω)": 3, "PV − setpoint (Ω)": 3, "GDM vs PV (ppm)": 0}).to_string())
